# World Airport Network

Starter notebook for the Kaggle dataset in this folder. It loads `airports.csv` with pandas and walks through quality checks and a first piece of analysis.

**Files:** `airports.csv`, `data_dictionary.csv`.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("airports.csv", low_memory=False)
df["routes"] = df["n_routes_out"].fillna(0) + df["n_routes_in"].fillna(0)
print(df.shape)
df.head(3)

## 2. Shape, types and missing values

In [ ]:
import pandas as pd

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("missing_%").head(15)

## 3. Numeric summary

In [ ]:
df.describe(include='number').T.round(2).head(20)

## 4. Busiest airports by route records

In [ ]:
df.nlargest(15, "routes")[["name", "city", "country", "iata", "routes",
                                     "n_destinations", "n_airlines"]].set_index("iata")

## 5. Hub classes

How many airports sit in each network role?

In [ ]:
df["hub_class"].value_counts().plot.bar(figsize=(8, 4))

## 6. Long-haul map

Airports with the longest non-stop routes.

In [ ]:
long = df.dropna(subset=["max_route_distance_km"]).nlargest(20, "max_route_distance_km")
long[["name", "city", "country", "max_route_distance_km", "farthest_destination"]].set_index("name")

## 7. Connectivity vs wealth

In [ ]:
sub = df.dropna(subset=["country_gdp_per_capita_usd", "routes_per_million_people"])
sub = sub[sub["routes"] > 0]
sub.plot.scatter(x="country_gdp_per_capita_usd", y="routes_per_million_people", logx=True, s=10, figsize=(8, 5))

## 8. Domestic vs international mix for the top 25 airports

In [ ]:
top = df.nlargest(25, "routes").set_index("iata")
(top[["n_domestic_routes", "n_routes_out"]]
   .assign(international=lambda t: t["n_routes_out"] - t["n_domestic_routes"])
   [["n_domestic_routes", "international"]]
   .plot.barh(stacked=True, figsize=(9, 7)))

## Next steps

See `KAGGLE.md` for analysis ideas, caveats and provenance.